# Stage 2: baseline samplers on MATH500 (PF / ePF)

Runs on a free Kaggle GPU. Before executing anything:

1. **Settings (right sidebar) -> Accelerator -> GPU T4 x2** (or P100).
2. **Settings -> Internet -> On** (needed to clone the repo and download models).
3. To run unattended (survives closing the browser tab): use **Save Version -> Save & Run All (Commit)** instead of running cells interactively. Free quota is ~30 GPU-hours/week; a session caps out around 9-12 hours.

This is the plan doc's hard gate (§6): reproduce standard particle filtering (PF) and Entropic Particle Filtering (ePF) on MATH500 with Qwen2.5-1.5B within ~2 accuracy points of published numbers, before trusting anything built on top. All four baselines (PF / beam / twisted-SMC-fixed / ePF) share one driver, `smc/sampler_llm.py::Sampler` -- this notebook only takes **PF and ePF** through real verification (this push's scope; beam/twisted-SMC are wired but deferred to a follow-up).

This notebook: (a) clones the repo, (b) runs the CPU-only `--dry-run` smoke test first (catches repo/env problems for free), (c) installs the GPU extras, (d) runs the `score_batch()`-vs-`score()` equivalence check -- a **prerequisite**, not optional, before trusting batched PRM scoring inside the Sampler loop, (e) a tiny real-GPU sanity check on 3 problems for each method, (f) only then the full 128-problem runs for PF and ePF.

**Memory note, different from Stage 1:** in Stage 1 the policy (vLLM) and the PRM were never resident on the GPU at the same time (generate everything first, explicitly free the policy, only then load the PRM). Here, `Sampler.propagate()` calls **both** every single global SMC step, so they must stay loaded simultaneously for the whole run. `run_stage2_baselines.py --gpu-memory-utilization` (default 0.5) leaves headroom for the 8-bit PRM alongside vLLM's reservation -- lower it further (e.g. 0.35-0.4) if you still see an OOM while the PRM loads.

Everything written under the cloned repo directory (which lives under `/kaggle/working/`) persists in the notebook's Output after a commit -- no extra copy step needed.

Resumable at the GLOBAL-SMC-STEP level: every problem's full particle state is checkpointed to disk after every single SMC step (finer than Stage 1's per-rollout checkpointing -- one MATH500 problem here can run dozens of steps). If a session drops mid-run, just re-run the same full-run cell -- a partially-solved problem resumes from its last checkpointed step rather than restarting, and a fully-completed problem is skipped entirely. Pass `--fresh` to ignore any existing checkpoint/result and start clean.

In [ ]:
!git clone https://github.com/qqdexqq/twisted-smc-llm.git
%cd twisted-smc-llm

In [ ]:
# CPU-only deps first (fast, small). Kaggle's base image already has
# pandas/numpy/etc; -e . picks up anything missing (pyarrow, math-verify, datasets).
!pip install -q -e .

In [ ]:
# Free correctness check before spending any GPU time or downloading any
# model weights -- same philosophy as every prior stage. If this fails,
# the problem is in the repo/environment, not in a model or the GPU.
# Exercises all four methods' wiring (only PF/ePF get real-GPU runs below,
# but this costs nothing extra to check now).
!python scripts/run_stage2_baselines.py --method pf   --manifest manifests/math500_128.jsonl --dry-run --limit 3 --fresh
!python scripts/run_stage2_baselines.py --method epf  --manifest manifests/math500_128.jsonl --dry-run --limit 3 --fresh

## GPU extras

Installs `torch`/`transformers`/`vllm`/`bitsandbytes`. Confirmed cause of one real failure mode on Colab: the platform's preinstalled `torch` already "satisfies" the unpinned `torch` dependency, so pip leaves it alone -- but `vllm` resolves to a recent release that needs a newer CUDA runtime, leaving two mismatched CUDA generations installed side by side. The install below passes `--upgrade` to force pip to actually reconcile torch's version against vllm's real requirement. If it still errors on a CUDA/torch mismatch afterward: **Run -> Restart Session**, then re-run from this cell (skip the git clone / CPU-deps cells above) -- a session that already imported the old torch won't pick up the upgrade without a restart.

In [ ]:
!pip install -q -e ".[gpu]" --upgrade

## Prerequisite: score_batch() vs score() equivalence check

`models/prm.py::QwenMathPRMScorer.score_batch()` (the real batched-forward-pass PRM scoring `Sampler` uses every step) was written and CPU-sanity-checked against the *logic* of this comparison, but never run against the actual model weights. Per the plan: this must be verified empirically before trusting it inside the Sampler loop, not assumed. If this prints `FAIL`, **stop here** -- do not run the cells below -- and debug `models/prm.py::_make_step_rewards` / padding / `pad_token_id` first.

In [ ]:
!python scripts/diagnose_prm.py --prm-8bit

## Tiny real-GPU sanity check (3 problems each)

Small, fast checks before spending the GPU-hours on the full 128-problem runs. Eyeball the printed per-problem lines: `steps=` should mostly be a handful (single digits to a few dozen), not immediately hitting `--max-particle-steps` every time, and at least some answers should come out `correct=True` on MATH500 with a real model (unlike `--dry-run`'s mock, which is always incorrect by construction).

In [ ]:
!python scripts/run_stage2_baselines.py --method pf --manifest manifests/math500_128.jsonl \
    --dataset-name math500_sanity3 --limit 3 \
    --generator Qwen/Qwen2.5-1.5B-Instruct --prm Qwen/Qwen2.5-Math-PRM-7B --prm-type qwen --prm-8bit \
    --n-particles 8 --horizon-steps 24 --gpu-memory-utilization 0.5 --seed 0 --fresh

In [ ]:
!python scripts/run_stage2_baselines.py --method epf --manifest manifests/math500_128.jsonl \
    --dataset-name math500_sanity3 --limit 3 \
    --generator Qwen/Qwen2.5-1.5B-Instruct --prm Qwen/Qwen2.5-Math-PRM-7B --prm-type qwen --prm-8bit \
    --n-particles 8 --horizon-steps 24 --gpu-memory-utilization 0.5 --seed 0 --fresh

## Full runs -- the hard gate

Only run these once the sanity checks above look right. 128 problems x 16 particles on a 1.5B model -- expect this to take a while (each of the up-to-`--horizon-steps` global steps is its own batched generation call + batched PRM forward pass across all live particles); keep an eye on the free-tier session time limit.

If a cell gets interrupted (session drop, disconnect, anything), just re-run it as-is -- it checkpoints after every global SMC step and will skip any problem that already has a `.result.json`, resuming any problem that only got partway through from its last checkpointed step rather than restarting.

**After both finish:** compare the printed `accuracy=` against the published PF/ePF numbers for Qwen2.5-1.5B on MATH500 cited in `docs/Adaptive_TSMC_Build_Plan.md` (§1.2/§6). Within ~2 accuracy points on both closes this push's scope -- the plan's own instruction is to stop and debug before trusting anything downstream if this doesn't hold.

In [ ]:
!python scripts/run_stage2_baselines.py --method pf --manifest manifests/math500_128.jsonl \
    --generator Qwen/Qwen2.5-1.5B-Instruct --prm Qwen/Qwen2.5-Math-PRM-7B --prm-type qwen --prm-8bit \
    --n-particles 16 --horizon-steps 64 --gpu-memory-utilization 0.5 --seed 0

In [ ]:
!python scripts/run_stage2_baselines.py --method epf --manifest manifests/math500_128.jsonl \
    --generator Qwen/Qwen2.5-1.5B-Instruct --prm Qwen/Qwen2.5-Math-PRM-7B --prm-type qwen --prm-8bit \
    --n-particles 16 --horizon-steps 64 --gpu-memory-utilization 0.5 --seed 0

## After this finishes

- Per-problem results (`results/stage2/math500_128/{pf,epf}/seed=0/*.result.json`) and the aggregated `results.parquet` are under `results/stage2/` -- gitignored on purpose (same rule as Stage 1's `data/`: belongs on a persistent volume, not in git). Commit this notebook's **Output** (via Save & Run All) or download the parquet files directly from Kaggle's Output tab if you want them off of Kaggle.
- Each result row also carries `compute_*` columns (tokens generated, PRM forward passes, resample events, bisection solves -- `eval/compute_accounting.py`) for the doc's §7.2 compute-accounting requirement.
- If PF/ePF accuracy lands within ~2 points of the published numbers: this push's hard gate is satisfied. Next per the plan: twisted-SMC-fixed and beam search's own real-GPU verification (deferred, structural wiring already in place), then Stage 3's actual novel method.
- If it doesn't: stop and debug before anything downstream is trusted -- the plan doc's own instruction. Likely first places to look: the `score_batch()` equivalence check's actual numbers (re-scroll up), the sanity-check cells' per-problem `steps=`/`correct=` output, and whether `--horizon-steps`/`--max-particle-steps` are cutting off particles before they reach a `\boxed{}` answer.